# DaTSCAN — Fase 7: auditoría del test y CV estratificada por protocolo

Este notebook separa dos preguntas:

- **CV principal:** desempeño cuando todos los protocolos conocidos están representados en entrenamiento y validación.
- **Stress test anterior:** desempeño al excluir por completo un protocolo durante el entrenamiento.

Primero asigna los 35 NIfTI sin etiqueta al cluster técnico más cercano, mide si están dentro del dominio conocido y busca imágenes duplicadas o casi idénticas. Después construye cinco folds estratificados por la combinación `protocol_cluster × target` y entrena exactamente la misma CNN 3D local de dos canales utilizada en la fase 4.

No usa las etiquetas del test ni modifica el modelo a partir de sus imágenes.

## 0. Dependencias

In [ ]:
# Descomente solo si falta alguna dependencia.
# %pip install nibabel joblib scipy torch numpy pandas scikit-learn matplotlib seaborn

## 1. Librerías y configuración

In [ ]:
from pathlib import Path
import os
import copy, gc, hashlib, random, time, warnings
import joblib
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy.spatial.distance import cdist
from sklearn.metrics import log_loss, roc_auc_score, brier_score_loss
from sklearn.model_selection import StratifiedGroupKFold, StratifiedShuffleSplit
from sklearn.neighbors import NearestNeighbors
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore',category=FutureWarning);sns.set_theme(style='whitegrid')
SEED=20260910
def seed_everything(seed=SEED):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
seed_everything();torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:',torch.__version__,'| dispositivo:',DEVICE)
if DEVICE.type=='cuda':print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
import sys
REPO_ROOT=Path.cwd().resolve()
if REPO_ROOT.name=='notebooks':REPO_ROOT=REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:sys.path.insert(0,str(REPO_ROOT))
from src.config import DATA_ROOT
NIFTI_DIR=DATA_ROOT/'niftis_utCGpHE'
PROJECT_DIR=DATA_ROOT/'latent_protocol_cv'
FOLDS_CSV=PROJECT_DIR/'outputs'/'train_protocol_folds.csv'
METADATA_CSV=PROJECT_DIR/'outputs'/'protocol_metadata_clustered.csv'
PIPELINE_FILE=PROJECT_DIR/'artifacts'/'protocol_cluster_pipeline.joblib'
PREPROCESS_DIR=PROJECT_DIR/'preprocessed_96x96x64_v2'
ROI_CSV=PROJECT_DIR/'roi_adaptive_v3'/'roi_features.csv'
OUTPUT_DIR=PROJECT_DIR/'cv_protocol_stratified_2ch_v1'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

LOCAL_SHAPE=(48,48,32);BATCH_SIZE=4;NUM_WORKERS=0;MAX_EPOCHS=40;PATIENCE=7
LEARNING_RATE=1e-3;WEIGHT_DECAY=1e-5;INNER_VALID_FRACTION=.12;N_SPLITS=5
SAMPLE_VOXELS=200_000;FOREGROUND_RELATIVE_THRESHOLD=.01
for p in (NIFTI_DIR,FOLDS_CSV,METADATA_CSV,PIPELINE_FILE,PREPROCESS_DIR,ROI_CSV):
    print(p,'| existe:',p.exists())
    if not p.exists():raise FileNotFoundError(p)
print('Salida nueva:',OUTPUT_DIR)

## 2. Identificación de los 35 estudios sin etiqueta

In [ ]:
def normalize_uid(value):
    value=str(value).strip()
    for suffix in ('.nii.gz','.nii'):
        if value.endswith(suffix):value=value[:-len(suffix)]
    return value

files=sorted(NIFTI_DIR.rglob('*.nii.gz'))+sorted(NIFTI_DIR.rglob('*.nii'))
all_images=pd.DataFrame({'uid':[normalize_uid(p.name) for p in files],'nifti_path':[str(p.resolve()) for p in files]})
if all_images.uid.duplicated().any():raise ValueError('UID duplicados entre NIfTI.')
train_folds=pd.read_csv(FOLDS_CSV)
test_images=all_images[~all_images.uid.isin(train_folds.uid)].reset_index(drop=True)
print('NIfTI totales:',len(all_images),'| entrenamiento:',len(train_folds),'| sin etiqueta:',len(test_images))
if len(test_images)!=35:print('ADVERTENCIA: se esperaban 35 estudios sin etiqueta.')
display(test_images.head())

## 3. Metadatos técnicos y asignación del protocolo del test

In [ ]:
def deterministic_sample(values,n,uid):
    flat=values.reshape(-1)
    if flat.size<=n:return flat
    seed=int(hashlib.sha256(uid.encode()).hexdigest()[:8],16)
    return flat[np.random.default_rng(seed).choice(flat.size,size=n,replace=False)]

def safe_float(value):
    try:return float(np.asarray(value).reshape(-1)[0])
    except (TypeError,ValueError,IndexError):return np.nan

def extract_one(path,uid):
    img=nib.load(path,mmap=True);header=img.header;shape=tuple(int(v) for v in img.shape)
    if len(shape)<3:raise ValueError(f'Menos de 3 dimensiones: {path}')
    zooms=tuple(float(v) for v in header.get_zooms()[:3]);affine=np.asarray(img.affine,float)
    data=np.asarray(img.dataobj,dtype=np.float32);data=np.nan_to_num(data,nan=0,posinf=0,neginf=0)
    sampled=deterministic_sample(data,SAMPLE_VOXELS,uid);nonzero=sampled[sampled>0];basis=nonzero if nonzero.size>=100 else sampled
    p=np.percentile(basis,[1,5,25,50,75,90,95,99,99.5]);threshold=max(0,FOREGROUND_RELATIVE_THRESHOLD*float(p[-1]))
    spatial=shape[:3];fov=tuple(spatial[i]*zooms[i] for i in range(3))
    return {'uid':uid,'nifti_path':str(Path(path).resolve()),'shape_x':spatial[0],'shape_y':spatial[1],'shape_z':spatial[2],
      'ndim':len(shape),'n_voxels':int(np.prod(spatial)),'spacing_x':zooms[0],'spacing_y':zooms[1],'spacing_z':zooms[2],
      'voxel_volume_mm3':float(np.prod(zooms)),'fov_x_mm':fov[0],'fov_y_mm':fov[1],'fov_z_mm':fov[2],
      'orientation':''.join(nib.aff2axcodes(affine)),'affine_det_sign':int(np.sign(np.linalg.det(affine[:3,:3]))),
      'qform_code':int(header['qform_code']),'sform_code':int(header['sform_code']),'dtype':str(header.get_data_dtype()),
      'scl_slope':safe_float(header.get('scl_slope',np.nan)),'scl_inter':safe_float(header.get('scl_inter',np.nan)),
      'intensity_min':float(sampled.min()),'intensity_max':float(sampled.max()),'intensity_mean':float(sampled.mean()),'intensity_std':float(sampled.std()),
      **{f'intensity_{name}':float(v) for name,v in zip(('p01','p05','p25','p50','p75','p90','p95','p99','p995'),p)},
      'nonzero_fraction':float(np.mean(sampled>0)),'foreground_fraction':float(np.mean(sampled>threshold))}

TEST_METADATA=OUTPUT_DIR/'test_protocol_metadata.csv'
if TEST_METADATA.exists():test_meta=pd.read_csv(TEST_METADATA);print('Reutilizado:',TEST_METADATA)
else:
    rows=[]
    for i,row in test_images.iterrows():
        rows.append(extract_one(row.nifti_path,row.uid));print(f'{i+1}/{len(test_images)}',end='\r')
    test_meta=pd.DataFrame(rows);test_meta.to_csv(TEST_METADATA,index=False)
print('Metadatos test:',test_meta.shape)

In [ ]:
artifact = joblib.load(PIPELINE_FILE)

preprocessor = artifact["preprocessor"]
pca = artifact["pca"]
clusterer = artifact["clusterer"]

numeric_features = artifact["numeric_features"]
categorical_features = artifact["categorical_features"]

train_meta = pd.read_csv(METADATA_CSV)


def transform_protocol_metadata(df):
    missing = (
        set(numeric_features + categorical_features)
        - set(df.columns)
    )

    if missing:
        raise KeyError(
            f"Faltan variables requeridas: {sorted(missing)}"
        )

    # Aplicamos directamente los transformadores ajustados.
    # Esto evita el error de compatibilidad del ColumnTransformer.
    numeric_transformer = (
        preprocessor.named_transformers_["numeric"]
    )

    categorical_transformer = (
        preprocessor.named_transformers_["categorical"]
    )

    x_numeric = numeric_transformer.transform(
        df[numeric_features]
    )

    x_categorical = categorical_transformer.transform(
        df[categorical_features]
    )

    if hasattr(x_numeric, "toarray"):
        x_numeric = x_numeric.toarray()

    if hasattr(x_categorical, "toarray"):
        x_categorical = x_categorical.toarray()

    x_preprocessed = np.hstack([
        np.asarray(x_numeric),
        np.asarray(x_categorical)
    ])

    return pca.transform(x_preprocessed)


x_train = transform_protocol_metadata(train_meta)
x_test = transform_protocol_metadata(test_meta)

test_meta["protocol_cluster"] = clusterer.predict(x_test)

# Distancia de cada estudio al centro del cluster asignado.
train_clusters = train_meta["protocol_cluster"].astype(int).to_numpy()
test_clusters = test_meta["protocol_cluster"].astype(int).to_numpy()

train_dist = np.linalg.norm(
    x_train - clusterer.cluster_centers_[train_clusters],
    axis=1
)

test_dist = np.linalg.norm(
    x_test - clusterer.cluster_centers_[test_clusters],
    axis=1
)

thresholds = (
    pd.DataFrame({
        "protocol_cluster": train_clusters,
        "distance": train_dist
    })
    .groupby("protocol_cluster")["distance"]
    .quantile(0.99)
)

test_meta["distance_to_centroid"] = test_dist

test_meta["train_cluster_q99_distance"] = (
    test_meta["protocol_cluster"].map(thresholds)
)

test_meta["outside_train_q99"] = (
    test_meta["distance_to_centroid"]
    > test_meta["train_cluster_q99_distance"]
)

test_meta.to_csv(
    OUTPUT_DIR / "test_protocol_assignment.csv",
    index=False
)

test_summary = (
    test_meta
    .groupby("protocol_cluster")
    .agg(
        n=("uid", "size"),
        outside_q99=("outside_train_q99", "sum"),
        median_distance=("distance_to_centroid", "median"),
        max_distance=("distance_to_centroid", "max")
    )
    .reset_index()
)

display(test_summary)
display(test_meta[test_meta["outside_train_q99"]])

if test_meta["outside_train_q99"].any():
    print(
        "ATENCIÓN: existen estudios test fuera del "
        "percentil 99 de su cluster asignado."
    )
else:
    print(
        "Los 35 estudios están dentro del dominio "
        "técnico conocido al umbral q99."
    )

## 4. Auditoría de duplicados y casi duplicados en volúmenes preprocesados

In [ ]:
def thumbnail(path):
    with np.load(path) as saved:v=saved['volume'].astype(np.float32)
    # 12×12×8 = 1,152 valores; normalización individual para comparar estructura.
    t=v[::8,::8,::8].reshape(-1);t=(t-t.mean())/(t.std()+1e-6)
    return t

train_folds['processed_path']=train_folds.uid.map(lambda u:str((PREPROCESS_DIR/f'{u}.npz').resolve()))
missing=train_folds[~train_folds.processed_path.map(lambda p:Path(p).exists())]
if len(missing):raise FileNotFoundError(f'Faltan {len(missing)} volúmenes preprocesados.')
THUMB_FILE=OUTPUT_DIR/'train_thumbnails.npy'
if THUMB_FILE.exists():thumbs=np.load(THUMB_FILE)
else:
    thumbs=np.stack([thumbnail(p) for p in train_folds.processed_path]).astype(np.float32);np.save(THUMB_FILE,thumbs)
neighbor_model=NearestNeighbors(n_neighbors=2,metric='cosine').fit(thumbs);dist,idx=neighbor_model.kneighbors(thumbs)
similarity=1-dist[:,1]
nearest=pd.DataFrame({'uid':train_folds.uid,'nearest_uid':train_folds.uid.iloc[idx[:,1]].to_numpy(),'cosine_similarity':similarity,
                      'target':train_folds.target,'nearest_target':train_folds.target.iloc[idx[:,1]].to_numpy()})
display(nearest.nlargest(20,'cosine_similarity'));nearest.to_csv(OUTPUT_DIR/'nearest_image_pairs.csv',index=False)

# Umbral conservador: los pares señalados se agrupan para impedir que crucen folds.
parent=np.arange(len(train_folds))
def find(a):
    while parent[a]!=a:parent[a]=parent[parent[a]];a=parent[a]
    return a
def union(a,b):
    a,b=find(a),find(b)
    if a!=b:parent[b]=a
for i in range(len(train_folds)):
    if similarity[i]>=.9995:union(i,int(idx[i,1]))
train_folds['duplicate_group']=[find(i) for i in range(len(train_folds))]
groups=train_folds.groupby('duplicate_group').size();print('Grupos con más de una imagen:',int((groups>1).sum()))
conflict=(train_folds.groupby('duplicate_group').target.nunique()>1)
if conflict.any():
    display(train_folds[train_folds.duplicate_group.isin(conflict[conflict].index)])
    print('ADVERTENCIA: existen imágenes casi idénticas con etiquetas diferentes.')

## 5. Folds estratificados simultáneamente por protocolo y etiqueta

In [ ]:
train_folds['stratum']=train_folds.protocol_cluster.astype(str)+'_'+train_folds.target.astype(str)
splitter=StratifiedGroupKFold(n_splits=N_SPLITS,shuffle=True,random_state=SEED)
train_folds['protocol_stratified_fold']=-1
for fold,(_,va) in enumerate(splitter.split(train_folds,y=train_folds.stratum,groups=train_folds.duplicate_group)):
    train_folds.loc[va,'protocol_stratified_fold']=fold
if (train_folds.groupby('duplicate_group').protocol_stratified_fold.nunique()>1).any():raise RuntimeError('Un grupo duplicado cruzó folds.')
counts=pd.crosstab([train_folds.protocol_cluster,train_folds.target],train_folds.protocol_stratified_fold)
display(counts)
if (counts==0).any().any():raise RuntimeError('Algún fold no contiene una combinación protocolo-etiqueta.')
summary=train_folds.groupby('protocol_stratified_fold').agg(n=('uid','size'),prevalence=('target','mean'),n_clusters=('protocol_cluster','nunique')).reset_index()
display(summary);train_folds.to_csv(OUTPUT_DIR/'train_protocol_stratified_folds.csv',index=False)
print('Diferencia máxima de prevalencia:',summary.prevalence.max()-summary.prevalence.min())

## 6. Manifiesto para la CNN local de dos canales

In [ ]:
roi=pd.read_csv(ROI_CSV);loc_cols=['uid','midline_x','left_y','right_y','z_peak']
manifest=train_folds.merge(roi[loc_cols],on='uid',how='left',validate='one_to_one')
manifest['processed_path']=manifest.uid.map(lambda u:str((PREPROCESS_DIR/f'{u}.npz').resolve()))
if not manifest.processed_path.map(lambda p:Path(p).exists()).all():raise FileNotFoundError('Faltan volúmenes preprocesados.')
if manifest[loc_cols[1:]].isna().any().any():raise ValueError('Faltan coordenadas ROI.')
print('Estudios:',len(manifest));display(manifest.groupby('protocol_stratified_fold').agg(n=('uid','size'),prevalence=('target','mean')))

## 7. Dataset, arquitectura y smoke test

In [ ]:
def crop_pad(volume,center,shape=LOCAL_SHAPE):
    output=np.zeros(shape,dtype=np.float32)
    center=np.rint(center).astype(int);starts=center-np.asarray(shape)//2
    src=[];dst=[]
    for start,size,source_size in zip(starts,shape,volume.shape):
        src_start,src_end=max(0,start),min(source_size,start+size)
        dst_start=max(0,-start);dst_end=dst_start+max(0,src_end-src_start)
        src.append(slice(src_start,src_end));dst.append(slice(dst_start,dst_end))
    output[tuple(dst)]=volume[tuple(src)]
    return output

class DaTSCANDataset(Dataset):
    def __init__(self,frame,augment=False):
        self.frame=frame.reset_index(drop=True);self.augment=augment
    def __len__(self):return len(self.frame)
    def __getitem__(self,index):
        row=self.frame.iloc[index]
        with np.load(row.processed_path) as saved:volume=saved['volume'].astype(np.float32)
        center=(float(row.midline_x),float((row.left_y+row.right_y)/2),float(row.z_peak))
        local=crop_pad(volume,center)
        asymmetry=np.abs(local-local[::-1,:,:])
        x=torch.from_numpy(np.stack([local,asymmetry],axis=0))
        if self.augment:
            # Reflexión izquierda-derecha: conserva la etiqueta y evita preferencia lateral.
            if torch.rand(())<0.5:x=torch.flip(x,dims=(1,))
            # Baseline: perturbación de intensidad mínima; se ampliará solo si aprende.
            scale=float(torch.empty(1).uniform_(0.97,1.03))
            x=x*scale
            x=torch.clamp(x,0,1)
        return x,torch.tensor(float(row.target),dtype=torch.float32),str(row.uid)

def make_loader(frame,augment,shuffle,batch_size=BATCH_SIZE):
    return DataLoader(DaTSCANDataset(frame,augment),batch_size=batch_size,shuffle=shuffle,
                      num_workers=NUM_WORKERS,pin_memory=(DEVICE.type=='cuda'),drop_last=False)

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self,in_channels,out_channels):
        super().__init__()
        # GroupNorm exige que out_channels sea divisible por groups.
        groups=max(g for g in (8,4,2,1) if out_channels%g==0)
        self.block=nn.Sequential(
            nn.Conv3d(in_channels,out_channels,3,stride=2,padding=1,bias=False),
            nn.GroupNorm(groups,out_channels),nn.SiLU(inplace=True),
            nn.Conv3d(out_channels,out_channels,3,padding=1,bias=False),
            nn.GroupNorm(groups,out_channels),nn.SiLU(inplace=True))
    def forward(self,x):return self.block(x)

class SmallCNN3D(nn.Module):
    def __init__(self):
        super().__init__()
        self.features=nn.Sequential(ConvBlock(2,16),ConvBlock(16,32),ConvBlock(32,64))
        self.head=nn.Sequential(nn.Linear(128,64),nn.SiLU(),nn.Dropout(0.25),nn.Linear(64,1))
    def forward(self,x):
        x=self.features(x)
        pooled=torch.cat([F.adaptive_avg_pool3d(x,1),F.adaptive_max_pool3d(x,1)],dim=1).flatten(1)
        return self.head(pooled).squeeze(1)

model=SmallCNN3D().to(DEVICE)
print('Parámetros:',sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
smoke_frame=manifest.sample(n=min(8,len(manifest)),random_state=SEED)
smoke_loader=make_loader(smoke_frame,augment=True,shuffle=False,batch_size=min(BATCH_SIZE,4))
x_smoke,y_smoke,_=next(iter(smoke_loader))
smoke_model=SmallCNN3D().to(DEVICE)
optimizer=torch.optim.AdamW(smoke_model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
optimizer.zero_grad(set_to_none=True)
logits=smoke_model(x_smoke.to(DEVICE));loss=F.binary_cross_entropy_with_logits(logits,y_smoke.to(DEVICE))
loss.backward();optimizer.step()
print('Entrada:',tuple(x_smoke.shape),'| salida:',tuple(logits.shape),'| loss:',float(loss.detach().cpu()))
fig,axes=plt.subplots(1,2,figsize=(8,4))
axes[0].imshow(x_smoke[0,0,:,:,LOCAL_SHAPE[2]//2].T,cmap='inferno',origin='lower',vmin=0,vmax=1)
axes[0].set_title('Canal 1: intensidad local')
axes[1].imshow(x_smoke[0,1,:,:,LOCAL_SHAPE[2]//2].T,cmap='magma',origin='lower',vmin=0,vmax=1)
axes[1].set_title('Canal 2: diferencia I-D')
for ax in axes:ax.axis('off')
plt.tight_layout();plt.show()
del smoke_model,optimizer,x_smoke,y_smoke,logits,loss;gc.collect()
if DEVICE.type=='cuda':torch.cuda.empty_cache()

## 8. Entrenamiento sin utilizar el fold externo para early stopping

In [ ]:
@torch.no_grad()
def predict_loader(model,loader):
    model.eval();predictions=[];targets=[];uids=[]
    for x,y,batch_uids in loader:
        logits=model(x.to(DEVICE,non_blocking=True))
        predictions.extend(torch.sigmoid(logits).cpu().numpy().tolist())
        targets.extend(y.numpy().tolist());uids.extend(list(batch_uids))
    return np.asarray(predictions),np.asarray(targets,dtype=int),uids

def run_outer_fold(train_frame,outer_valid_frame,scheme,fold):
    seed_everything(SEED+fold)
    splitter=StratifiedShuffleSplit(n_splits=1,test_size=INNER_VALID_FRACTION,random_state=SEED+fold)
    inner_train_idx,inner_valid_idx=next(splitter.split(train_frame,train_frame.target))
    inner_train=train_frame.iloc[inner_train_idx];inner_valid=train_frame.iloc[inner_valid_idx]
    train_loader=make_loader(inner_train,True,True)
    inner_loader=make_loader(inner_valid,False,False)
    outer_loader=make_loader(outer_valid_frame,False,False)
    model=SmallCNN3D().to(DEVICE)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='min',factor=0.5,patience=2,min_lr=1e-6)
    scaler=torch.cuda.amp.GradScaler(enabled=(DEVICE.type=='cuda'))
    best_loss=np.inf;best_state=None;best_epoch=0;wait=0;history=[]
    for epoch in range(1,MAX_EPOCHS+1):
        model.train();running_loss=0.0;n_seen=0
        for x,y,_ in train_loader:
            x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(DEVICE.type=='cuda')):
                logits=model(x);batch_loss=F.binary_cross_entropy_with_logits(logits,y)
            scaler.scale(batch_loss).backward()
            scaler.unscale_(optimizer);torch.nn.utils.clip_grad_norm_(model.parameters(),5.0)
            scaler.step(optimizer);scaler.update()
            running_loss+=float(batch_loss.detach().cpu())*len(y);n_seen+=len(y)
        inner_pred,inner_y,_=predict_loader(model,inner_loader)
        inner_pred=np.clip(inner_pred,1e-6,1-1e-6);inner_loss=log_loss(inner_y,inner_pred)
        scheduler.step(inner_loss)
        history.append({'scheme':scheme,'fold':fold,'epoch':epoch,'train_loss':running_loss/n_seen,
                        'inner_logloss':inner_loss,'inner_auc':roc_auc_score(inner_y,inner_pred),
                        'learning_rate':optimizer.param_groups[0]['lr']})
        print(f'{scheme} F{fold} E{epoch:02d} | train={running_loss/n_seen:.4f} | inner={inner_loss:.4f}')
        if inner_loss<best_loss-1e-4:
            best_loss=inner_loss;best_epoch=epoch;best_state=copy.deepcopy(model.state_dict());wait=0
        else:
            wait+=1
            if wait>=PATIENCE:break
    model.load_state_dict(best_state)
    outer_pred,outer_y,outer_uids=predict_loader(model,outer_loader)
    outer_pred=np.clip(outer_pred,1e-6,1-1e-6)
    metrics={'scheme':scheme,'fold':fold,'n_train':len(train_frame),'n_valid':len(outer_valid_frame),
             'best_epoch':best_epoch,'inner_best_logloss':best_loss,
             'logloss':log_loss(outer_y,outer_pred),'auc':roc_auc_score(outer_y,outer_pred),
             'brier':brier_score_loss(outer_y,outer_pred),'valid_prevalence':float(outer_y.mean())}
    return model,pd.DataFrame(history),pd.DataFrame({'uid':outer_uids,'target':outer_y,'prediction':outer_pred}),metrics


In [ ]:
def run_cv(scheme,splits):
    oof=np.full(len(manifest),np.nan);metric_rows=[]
    uid_to_index={uid:i for i,uid in enumerate(manifest.uid)}
    for fold,(train_idx,valid_idx) in enumerate(splits):
        pred_path=OUTPUT_DIR/f'{scheme}_fold{fold}_predictions.csv'
        if pred_path.exists():
            fold_pred=pd.read_csv(pred_path);reused=True
            valid_uids=set(manifest.iloc[valid_idx].uid)
            if set(fold_pred.uid)!=valid_uids:raise ValueError(f'Predicciones incompatibles en {pred_path}')
            y_fold=fold_pred.target.astype(int).to_numpy();p_fold=fold_pred.prediction.to_numpy()
            metrics={'scheme':scheme,'fold':fold,'n_train':len(train_idx),'n_valid':len(valid_idx),'best_epoch':np.nan,
                     'inner_best_logloss':np.nan,'logloss':log_loss(y_fold,p_fold),'auc':roc_auc_score(y_fold,p_fold),
                     'brier':brier_score_loss(y_fold,p_fold),'valid_prevalence':float(y_fold.mean()),'reused':True}
        else:
            model,history,fold_pred,metrics=run_outer_fold(manifest.iloc[train_idx],manifest.iloc[valid_idx],scheme,fold)
            torch.save(model.state_dict(),OUTPUT_DIR/f'{scheme}_fold{fold}.pt')
            history.to_csv(OUTPUT_DIR/f'{scheme}_fold{fold}_history.csv',index=False)
            fold_pred.to_csv(pred_path,index=False);metrics['reused']=False
            del model;gc.collect()
            if DEVICE.type=='cuda':torch.cuda.empty_cache()
        for uid,pred in zip(fold_pred.uid,fold_pred.prediction):oof[uid_to_index[uid]]=pred
        metric_rows.append(metrics);pd.DataFrame(metric_rows).to_csv(OUTPUT_DIR/f'{scheme}_fold_metrics_checkpoint.csv',index=False)
        display(pd.DataFrame([metrics]))
    if np.isnan(oof).any():raise RuntimeError('OOF incompleto.')
    overall={'scheme':scheme,'n':len(oof),'logloss_oof':log_loss(manifest.target,oof),
             'auc_oof':roc_auc_score(manifest.target,oof),'brier_oof':brier_score_loss(manifest.target,oof)}
    pd.DataFrame({'uid':manifest.uid,'target':manifest.target,'protocol_cluster':manifest.protocol_cluster,
                  'fold':manifest.protocol_stratified_fold,'prediction':oof}).to_csv(OUTPUT_DIR/f'{scheme}_oof.csv',index=False)
    pd.DataFrame(metric_rows).to_csv(OUTPUT_DIR/f'{scheme}_fold_metrics.csv',index=False)
    return oof,pd.DataFrame(metric_rows),overall

## 9. Ejecutar la CV estratificada por protocolo

In [ ]:
splits=[]
for fold in sorted(manifest.protocol_stratified_fold.unique()):
    va=np.where(manifest.protocol_stratified_fold.to_numpy()==fold)[0]
    tr=np.where(manifest.protocol_stratified_fold.to_numpy()!=fold)[0]
    splits.append((tr,va))
stratified_output=run_cv('protocol_stratified',splits)
oof,fold_metrics,overall=stratified_output
display(fold_metrics);display(pd.DataFrame([overall]))

## 10. Diagnóstico OOF por protocolo y comparación con el stress test

In [ ]:
rows=[]
for cluster,indices in manifest.groupby('protocol_cluster').groups.items():
    idx=np.asarray(list(indices));y=manifest.target.to_numpy()[idx];p=oof[idx]
    rows.append({'protocol_cluster':cluster,'n':len(idx),'prevalence':y.mean(),'logloss':log_loss(y,p),
                 'auc':roc_auc_score(y,p),'brier':brier_score_loss(y,p)})
by_cluster=pd.DataFrame(rows);display(by_cluster);by_cluster.to_csv(OUTPUT_DIR/'protocol_stratified_metrics_by_cluster.csv',index=False)

STRESS_OOF=PROJECT_DIR/'cnn3d_local_2ch_v1'/'protocol_grouped_oof.csv'
comparison=[{'scheme':'protocol_stratified',**{k:v for k,v in overall.items() if k!='scheme'}}]
if STRESS_OOF.exists():
    stress=pd.read_csv(STRESS_OOF);comparison.append({'scheme':'leave_protocol_out','n':len(stress),
      'logloss_oof':log_loss(stress.target,stress.prediction),'auc_oof':roc_auc_score(stress.target,stress.prediction),
      'brier_oof':brier_score_loss(stress.target,stress.prediction)})
comparison=pd.DataFrame(comparison);display(comparison);comparison.to_csv(OUTPUT_DIR/'comparison_primary_vs_stress.csv',index=False)

fig,axes=plt.subplots(1,2,figsize=(12,4))
sns.barplot(data=by_cluster,x='protocol_cluster',y='logloss',ax=axes[0],color='#386641')
sns.barplot(data=by_cluster,x='protocol_cluster',y='auc',ax=axes[1],color='#6A4C93');axes[1].axhline(.5,color='crimson',ls='--')
plt.tight_layout();plt.savefig(OUTPUT_DIR/'metrics_by_cluster_protocol_stratified.png',dpi=170,bbox_inches='tight');plt.show()

## Interpretación final

- Si los 35 estudios de prueba caen dentro de clusters conocidos, la CV estratificada por protocolo es la estimación principal más cercana al escenario de competencia.
- La CV leave-protocol-out permanece como stress test de transporte hacia adquisiciones completamente nuevas.
- Una diferencia grande entre ambas no es contradicción: cuantifica el costo del cambio de dominio.
- La estratificación conjunta impide que un protocolo o una clase desaparezcan de un fold.
- Los grupos de imágenes casi idénticas, si existen, permanecen en un único fold.